In [8]:
import os
import sys
sys.path.append('/host/d/Github')  ### remove this if not needed!
import numpy as np
import nibabel as nb
import pandas as pd
import random
import torch
from tqdm import tqdm
import torch.backends.cudnn as cudnn
import argparse
 
from whole_heart_segmentation_ZC.utils.model_util import *
from whole_heart_segmentation_ZC.segment_anything.model import build_model 
from whole_heart_segmentation_ZC.utils.save_utils import *
from whole_heart_segmentation_ZC.utils.config_util import Config
from whole_heart_segmentation_ZC.utils.misc import NativeScalerWithGradNormCount as NativeScaler

import whole_heart_segmentation_ZC.functions_collection as ff
import whole_heart_segmentation_ZC.Data_processing as Data_processing
import whole_heart_segmentation_ZC.Build_lists.Build_list as Build_list
import whole_heart_segmentation_ZC.data_loader.generator_4classes as generator
from torch.utils.data import Dataset, DataLoader

main_path='/host/d/projects/WHS/' ### change to your main path

### step 1: define parameters for this experiment

In [9]:
trial_name = 'WHS_processed_v1_3classes'
output_dir = os.path.join(main_path, 'models', trial_name) # change to your output dir
ff.make_folder([output_dir])

#### important! define your pretrained model
pretrained_model = os.path.join(output_dir, 'models/model-195.pth')
print('pretrained model path: ', pretrained_model)

pretrained model path:  /host/d/projects/WHS/models/WHS_processed_v1_3classes/models/model-195.pth


In [10]:
# many important parameters, focus on ones that I comment with ###!!

def get_args_parser(img_size = 256, num_classes = 4, slice_num = 5, pretrained_model = None, original_sam = None, start_epoch = None, total_training_epochs = 1000, save_model_every = 1,  vit_type = "vit_b"):
    parser = argparse.ArgumentParser('SAM fine-tuning', add_help=True)

    # img size
    parser.add_argument('--img_size', default=img_size, type=int)  ## !!

    ## augmentation
    parser.add_argument('--augment_frequency', default= 0.5, type=float) ## !! ise a proper frequency

    ## segmentation classes
    parser.add_argument('--num_classes', type=int, default=num_classes) ## !!

    ## pretrained sam
    parser.add_argument('--resume', default = original_sam) ##!!

    # for training
    parser.add_argument('--total_training_epochs', default = total_training_epochs, type=int)
    parser.add_argument('--accum_iter', default=1, type=int, help='Accumulate gradient iterations (for increasing the effective batch size under memory constraints)') ##!!
    parser.add_argument('--save_model_file_every_N_epoch', default=save_model_every, type = int)  ## !!
    parser.add_argument('--batch_size', default=1, type=int, help='Batch size per GPU (effective batch size is batch_size * accum_iter * # gpus')  ## !!
    parser.add_argument('--weight_decay', type=float, default=0.05, help='weight decay (default: 0.05)')
    parser.add_argument('--lr', type=float, default=1e-4, metavar='LR', help='base learning rate: absolute_lr = base_lr * total_batch_size / 256') ## !!
    parser.add_argument('--lr_update_every_N_epoch', default=100, type = int) ## !!
    parser.add_argument('--lr_decay_gamma', default=0.95)
    parser.add_argument('--warmup_epochs', type=int, default=10, metavar='N', help='epochs to warmup LR')

    # for loss calculation
    parser.add_argument('--DICE_only_present_mask', default=True, type=bool, help='If True, calculate DICE loss only for classes present in the ground truth')
    parser.add_argument('--CE', default='focal', type=str, help='Type of cross entropy loss to use: "focal" or "standard"')
    parser.add_argument('--loss_weights', default = [1,1] )  #### !! weighting for loss function [BCE, Dice]

    if start_epoch == None:
        parser.add_argument('--start_epoch', default=1, type=int, metavar='N', help='start epoch')
    else:
        parser.add_argument('--start_epoch', default= start_epoch, type=int, metavar='N', help='start epoch')

    # standard
    parser.add_argument('--text_prompt', default = False)
    parser.add_argument('--box_prompt', default= False) 
    parser.add_argument('--pretrained_model', default = pretrained_model)
    
    parser.add_argument('--validation', default=False) ## !!
    parser.add_argument('--cross_frame_attention', default=False) # False

    parser.add_argument('--model_type', type=str, default='sam')
    parser.add_argument('--n_gpu', type=int, default=1, help='total gpu') 
    parser.add_argument('--use_amp', action='store_true', help='If activated, adopt mixed precision for acceleration')
    parser.add_argument("--config", help="Path to the training config file.", default="configs/config.yaml",)

    parser.add_argument('--seed', default=1234, type=int)   
    parser.add_argument('--input_type', type=str, default='2DT') #has to be 2DT
    parser.add_argument('--vit_type', type=str, default=vit_type)
    parser.add_argument('--slice_num', default=slice_num, type=int) 
                        

    parser.add_argument('--turn_zero_seg_slice_into', default=10, type=int)
 
    return parser


In [11]:
# define the original sam model weights (you should download it from online to your local path)
original_sam = '/host/d/Data/pretrained_SAM_weights/sam_vit_b.pth'  # change to your original sam model path

slice_num = 15

args = get_args_parser(img_size = 256, ## important !! need to change based on your dataset
        num_classes = 4, ## important !! need to change based on your dataset
        slice_num = slice_num,
        pretrained_model = pretrained_model, 
        original_sam = original_sam,  
        vit_type = "vit_b")

args = args.parse_args([])

# some other settings
cfg = Config(args.config)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
cudnn.benchmark = True

### step 2: define data you want to predict on and corresponding data generator


In [12]:
# # change the excel path to your own path
patient_list_spreadsheet = os.path.join('/host/d/Data/WHS/Patient_lists','train_val_path_list_processed_v1.xlsx')
build_sheet =  Build_list.Build(patient_list_spreadsheet)
# test
batch_list, modality_list, center_list, patient_id_list, size_x_list, size_y_list, size_z_list, img_file_list, seg_file_list = build_sheet.__build__(batch_list = [4])  

print('test ', len(img_file_list), ' one case example: ', img_file_list[4])


test  18  one case example:  /host/d/Data/WHS/current_challenge/processed_data_v1/CT/centerA/1020/image.nii.gz


### predict

In [13]:
# data_loader_pred = torch.utils.data.DataLoader(generator_pred, batch_size = 1, shuffle = False, pin_memory = True, num_workers = 0)

model = build_model(args, device)#skip_nameing = True, chunk = np.shape(np.zeros(0)))

# load the pretrained model
if args.pretrained_model is not None:
    print('loading pretrained model : ', args.pretrained_model)
    finetune_checkpoint = torch.load(args.pretrained_model)
    model.load_state_dict(finetune_checkpoint["model"])
else:
    ValueError('no pretrained model specified for prediction!')

Important! text prompt: False
Important! box prompt: False
loading pretrained model :  /host/d/projects/WHS/models/WHS_processed_v1_3classes/models/model-195.pth


In [14]:
sliding_windows = False
for i in range(0,patient_id_list.shape[0]):
    modality = modality_list[i]
    center = center_list[i]
    patient_id = patient_id_list[i]
    image_file = img_file_list[i]
    seg_file = seg_file_list[i]
    print('doing modality: ', modality, ', center: ', center, ', patient_id: ', patient_id)
    save_folder_patient = os.path.join(output_dir, 'predictions', modality, center, patient_id)
    ff.make_folder([os.path.join(output_dir,'predictions'), os.path.join(output_dir,'predictions', modality), os.path.join(output_dir,'predictions', modality, center), save_folder_patient])
    

    # not doing sliding windows
    if sliding_windows == False:
        # load a seg file
        seg_gt = nb.load(seg_file)
        affine = seg_gt.affine
        seg_gt = seg_gt.get_fdata()
        shape = seg_gt.shape
        size_x, size_y, size_z = shape

        how_many_slices_set_per_case = size_z // args.slice_num
        print('size_z: ', size_z, ', how_many_slices_set_per_case: ', how_many_slices_set_per_case)

        slice_range = [0, args.slice_num*how_many_slices_set_per_case]
        # save the ground truth (with new slice range)
        seg_gt = seg_gt[:, :, slice_range[0]:slice_range[1]]
        ii_new = np.zeros_like(seg_gt)
        ii_new[seg_gt==500] = 1
        ii_new[seg_gt==600] = 2
        ii_new[seg_gt==205] = 3
        seg_gt = np.copy(ii_new)
        nb.save(nb.Nifti1Image(seg_gt, affine), os.path.join(save_folder_patient, 'seg_gt.nii.gz'))

        img_gt = nb.load(image_file).get_fdata()[:, :, slice_range[0]:slice_range[1]]
        nb.save(nb.Nifti1Image(img_gt, affine), os.path.join(save_folder_patient, 'img_gt.nii.gz'))

        
        # define generator
        generator_test = generator.Dataset_CMR_Simple(
                image_file_list = np.asarray([image_file]),
                seg_file_list = np.asarray([seg_file]),
            
                args = args,
                how_many_slices_set_per_case = how_many_slices_set_per_case,
                slice_range = slice_range,

                shuffle = False,
                image_normalization = True,
                augment = False,
                augment_frequency = 0, )


        # do the predictions
        pred_seg_final = np.zeros_like(seg_gt)
        # pred_seg_final = np.zeros([args.img_size, args.img_size, seg_gt.shape[2]])

        data_loader_pred = torch.utils.data.DataLoader(generator_test, batch_size = 1, shuffle = False, pin_memory = True, num_workers = 0)
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                                    
                # do the prediction for each slice (2D+T) one by one
                for data_iter_step, batch in tqdm(enumerate(data_loader_pred)):

                    image_filename = batch["image_filename"][0]
                    patient_id = os.path.basename(os.path.dirname(image_filename))
                    

                    batch["image"]= batch["image"].cuda()
                            
                    output = model(batch, args.img_size)

                    torch.cuda.synchronize()

                    pred_seg = np.rollaxis(output["masks"].argmax(1).detach().cpu().numpy(), 0, 3)
                    # print('pred_seg shape:', pred_seg.shape)

                    pred_seg = Data_processing.crop_or_pad(pred_seg, target_size = [size_x, size_y, pred_seg.shape[2]], padding_value = 0)
                    # print('pred_seg shape after crop_or_pad:', pred_seg.shape)

                    pred_seg_final[:,:, batch["slice_range"][0]:batch["slice_range"][1]] = pred_seg

        nb.save(nb.Nifti1Image(pred_seg_final, affine), os.path.join(save_folder_patient, 'seg_pred.nii.gz'))


    ########### doing sliding window
    else:
        # load a seg file
        seg_gt = nb.load(seg_file)
        affine = seg_gt.affine
        seg_gt = seg_gt.get_fdata()
        shape = seg_gt.shape
        size_x, size_y, size_z = shape
        slice_range = [0, size_z]

        # save the ground truth (with new slice range)
        seg_gt = seg_gt[:, :, slice_range[0]:slice_range[1]]
        ii_new = np.zeros_like(seg_gt)
        ii_new[seg_gt==500] = 1
        ii_new[seg_gt==600] = 2
        ii_new[seg_gt==205] = 3
        seg_gt = np.copy(ii_new)
        nb.save(nb.Nifti1Image(seg_gt, affine), os.path.join(save_folder_patient, 'seg_gt.nii.gz'))

        img_gt = nb.load(image_file).get_fdata()[:, :, slice_range[0]:slice_range[1]]
        nb.save(nb.Nifti1Image(img_gt, affine), os.path.join(save_folder_patient, 'img_gt.nii.gz'))

        # define generator
        generator_test = generator.Dataset_CMR_Simple_sliding(
                image_file_list = np.asarray([image_file]),
                seg_file_list = np.asarray([seg_file]),
            
                args = args,
             
                slice_range = [0, size_z],

                shuffle = False,
                image_normalization = True,
                augment = False,
                augment_frequency = 0, )
        
         # do the predictions
        pred_seg_final = np.zeros((args.num_classes, args.img_size, args.img_size, seg_gt.shape[2]))

        data_loader_pred = torch.utils.data.DataLoader(generator_test, batch_size = 1, shuffle = False, pin_memory = True, num_workers = 0)
        with torch.no_grad():
            with torch.cuda.amp.autocast():
                                    
                # do the prediction for each slice (2D+T) one by one
                for data_iter_step, batch in tqdm(enumerate(data_loader_pred)):

                    image_filename = batch["image_filename"][0]
                    patient_id = os.path.basename(os.path.dirname(image_filename))
                    

                    batch["image"]= batch["image"].cuda()
                            
                    output = model(batch, args.img_size)

                    torch.cuda.synchronize()

                    pred_seg = np.rollaxis(output["masks"].detach().cpu().numpy(), 0, 4)
                    pred_seg_final[:, :, :, batch["slice_range"][0]:batch["slice_range"][1]] = pred_seg
        # do average
        counts = np.array(batch["counts"], dtype=np.float32)
        pred_seg_final = pred_seg_final / counts[None,None,None,:]
        pred_seg_final = np.argmax(pred_seg_final, axis=0).astype(np.int16)

        pred_seg_final = Data_processing.crop_or_pad(pred_seg_final, target_size = [size_x, size_y, pred_seg_final.shape[2]], padding_value = 0)

        nb.save(nb.Nifti1Image(pred_seg_final, affine), os.path.join(save_folder_patient, 'seg_pred.nii.gz'))
        

        


doing modality:  CT , center:  centerA , patient_id:  1002
windows: [[0, 15], [1, 16], [2, 17], [3, 18], [4, 19], [5, 20], [6, 21], [7, 22], [8, 23], [9, 24], [10, 25], [11, 26], [12, 27], [13, 28], [14, 29], [15, 30], [16, 31], [17, 32], [18, 33], [19, 34], [20, 35], [21, 36], [22, 37], [23, 38], [24, 39], [25, 40], [26, 41], [27, 42], [28, 43], [29, 44], [30, 45], [31, 46], [32, 47], [33, 48], [34, 49], [35, 50], [36, 51], [37, 52], [38, 53], [39, 54], [40, 55], [41, 56], [42, 57], [43, 58], [44, 59], [45, 60], [46, 61], [47, 62], [48, 63], [49, 64], [50, 65], [51, 66], [52, 67], [53, 68], [54, 69], [55, 70], [56, 71], [57, 72], [58, 73], [59, 74], [60, 75], [61, 76], [62, 77], [63, 78], [64, 79], [65, 80], [66, 81], [67, 82], [68, 83], [69, 84], [70, 85], [71, 86], [72, 87], [73, 88], [74, 89], [75, 90], [76, 91], [77, 92], [78, 93], [79, 94], [80, 95], [81, 96], [82, 97], [83, 98], [84, 99], [85, 100]]


86it [00:26,  3.30it/s]
/tmp/ipykernel_34393/3420537077.py:146: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  counts = np.array(batch["counts"], dtype=np.float32)
/tmp/ipykernel_34393/3420537077.py:146: DeprecationWarning: setting an array element with a sequence. This was supported in some cases where the elements are arrays with a single element. For example `np.array([1, np.array([2])], dtype=int)`. In the future this will raise the same ValueError as `np.array([1, [2]], dtype=int)`.
  counts = np.array(batch["counts"], dtype=np.float32)


doing modality:  CT , center:  centerA , patient_id:  1007
windows: [[0, 15], [1, 16], [2, 17], [3, 18], [4, 19], [5, 20], [6, 21], [7, 22], [8, 23], [9, 24], [10, 25], [11, 26], [12, 27], [13, 28], [14, 29], [15, 30], [16, 31], [17, 32], [18, 33], [19, 34], [20, 35], [21, 36], [22, 37], [23, 38], [24, 39], [25, 40], [26, 41], [27, 42], [28, 43], [29, 44], [30, 45], [31, 46], [32, 47], [33, 48], [34, 49], [35, 50], [36, 51], [37, 52], [38, 53], [39, 54], [40, 55], [41, 56], [42, 57], [43, 58], [44, 59], [45, 60], [46, 61], [47, 62], [48, 63], [49, 64], [50, 65], [51, 66], [52, 67], [53, 68], [54, 69], [55, 70], [56, 71], [57, 72], [58, 73], [59, 74], [60, 75], [61, 76], [62, 77], [63, 78], [64, 79], [65, 80], [66, 81], [67, 82], [68, 83], [69, 84], [70, 85], [71, 86], [72, 87], [73, 88], [74, 89], [75, 90], [76, 91], [77, 92], [78, 93], [79, 94], [80, 95], [81, 96], [82, 97], [83, 98], [84, 99], [85, 100], [86, 101]]


87it [00:22,  3.95it/s]


doing modality:  CT , center:  centerA , patient_id:  1008
windows: [[0, 15], [1, 16], [2, 17], [3, 18], [4, 19], [5, 20], [6, 21], [7, 22], [8, 23], [9, 24], [10, 25], [11, 26], [12, 27], [13, 28], [14, 29], [15, 30], [16, 31], [17, 32], [18, 33], [19, 34], [20, 35], [21, 36], [22, 37], [23, 38], [24, 39], [25, 40], [26, 41], [27, 42], [28, 43], [29, 44], [30, 45], [31, 46], [32, 47], [33, 48], [34, 49], [35, 50], [36, 51], [37, 52], [38, 53], [39, 54], [40, 55], [41, 56], [42, 57], [43, 58], [44, 59], [45, 60], [46, 61], [47, 62], [48, 63], [49, 64], [50, 65], [51, 66], [52, 67], [53, 68], [54, 69], [55, 70], [56, 71], [57, 72], [58, 73], [59, 74], [60, 75], [61, 76], [62, 77], [63, 78], [64, 79], [65, 80], [66, 81], [67, 82], [68, 83], [69, 84], [70, 85], [71, 86], [72, 87], [73, 88], [74, 89], [75, 90], [76, 91], [77, 92], [78, 93]]


...

doing modality:  CT , center:  centerB , patient_id:  2008
windows: [[0, 15], [1, 16], [2, 17], [3, 18], [4, 19], [5, 20], [6, 21], [7, 22], [8, 23], [9, 24], [10, 25], [11, 26], [12, 27], [13, 28], [14, 29], [15, 30], [16, 31], [17, 32], [18, 33], [19, 34], [20, 35], [21, 36], [22, 37], [23, 38], [24, 39], [25, 40], [26, 41], [27, 42], [28, 43], [29, 44], [30, 45], [31, 46], [32, 47], [33, 48], [34, 49], [35, 50], [36, 51], [37, 52], [38, 53], [39, 54], [40, 55], [41, 56], [42, 57], [43, 58], [44, 59], [45, 60], [46, 61], [47, 62], [48, 63], [49, 64], [50, 65], [51, 66], [52, 67], [53, 68], [54, 69], [55, 70], [56, 71], [57, 72], [58, 73], [59, 74], [60, 75], [61, 76], [62, 77], [63, 78], [64, 79], [65, 80], [66, 81], [67, 82], [68, 83], [69, 84], [70, 85], [71, 86], [72, 87], [73, 88], [74, 89], [75, 90], [76, 91], [77, 92], [78, 93], [79, 94], [80, 95], [81, 96], [82, 97], [83, 98], [84, 99], [85, 100], [86, 101], [87, 102], [88, 103], [89, 104], [90, 105], [91, 106], [92, 107], [93

146it [00:45,  3.22it/s]


doing modality:  CT , center:  centerB , patient_id:  2012
windows: [[0, 15], [1, 16], [2, 17], [3, 18], [4, 19], [5, 20], [6, 21], [7, 22], [8, 23], [9, 24], [10, 25], [11, 26], [12, 27], [13, 28], [14, 29], [15, 30], [16, 31], [17, 32], [18, 33], [19, 34], [20, 35], [21, 36], [22, 37], [23, 38], [24, 39], [25, 40], [26, 41], [27, 42], [28, 43], [29, 44], [30, 45], [31, 46], [32, 47], [33, 48], [34, 49], [35, 50], [36, 51], [37, 52], [38, 53], [39, 54], [40, 55], [41, 56], [42, 57], [43, 58], [44, 59], [45, 60], [46, 61], [47, 62], [48, 63], [49, 64], [50, 65], [51, 66], [52, 67], [53, 68], [54, 69], [55, 70], [56, 71], [57, 72], [58, 73], [59, 74], [60, 75], [61, 76], [62, 77], [63, 78], [64, 79], [65, 80], [66, 81], [67, 82], [68, 83], [69, 84], [70, 85], [71, 86], [72, 87], [73, 88], [74, 89], [75, 90], [76, 91], [77, 92], [78, 93], [79, 94], [80, 95], [81, 96], [82, 97], [83, 98], [84, 99], [85, 100], [86, 101], [87, 102], [88, 103], [89, 104], [90, 105], [91, 106], [92, 107], [93

10it [00:03,  2.65it/s]


KeyboardInterrupt: 